In [24]:
import os, getpass
import mysql.connector
import pandas as pd
import plotly.express as px

# Connect to MySQL server
mydb = mysql.connector.connect(
 host="localhost",
 user="root",
 password=os.getenv("MYSQL_PASSWORD") or getpass.getpass("MySQL password: "),
 database="indian_agriculture",)

mycursor = mydb.cursor(buffered=True)
print(mydb.is_connected())

True


In [25]:
def run_query(query):
    """Run SQL and return a DataFrame; column names come from the query itself."""
    mycursor.execute(query)
    cols = [c[0] for c in mycursor.description]
    df = pd.DataFrame(mycursor.fetchall(), columns=cols)
    for c in df.columns:                    
        try:
            df[c] = pd.to_numeric(df[c])
        except (ValueError, TypeError):
            pass
    return df

1.Year-wise Trend of Rice Production Across States (Top 3)

In [26]:
q1 = """
SELECT YEAR AS year, STATE_NAME AS state,
       SUM(RICE_PRODUCTION_1000_TONS) AS rice_production
FROM agri_data
WHERE STATE_NAME IN (
    SELECT STATE_NAME FROM (
        SELECT STATE_NAME
        FROM agri_data
        GROUP BY STATE_NAME
        ORDER BY SUM(RICE_PRODUCTION_1000_TONS) DESC
        LIMIT 3
    ) AS top3_states
)
GROUP BY YEAR, STATE_NAME
ORDER BY YEAR, rice_production DESC
"""
df1 = run_query(q1)
display(df1.head(10))
px.line(df1, x="year", y="rice_production", color="state",
        title="Rice production (1000 tons): top 3 states").show()

,year,state,rice_production
0,1966,West Bengal,4819.40
1,1966,Uttar Pradesh,2473.50
2,1966,Punjab,338.00
3,1967,West Bengal,4865.77
4,1967,Uttar Pradesh,2969.00
5,1967,Punjab,415.00
6,1968,West Bengal,5461.49
7,1968,Uttar Pradesh,2607.10
8,1968,Punjab,469.50
9,1969,West Bengal,5362.82


2.Top 5 Districts by Wheat Yield Increase Over the Last 5 Years

In [27]:
q2 = """
SELECT a.DIST_NAME AS district, a.STATE_NAME AS state,
       b.WHEAT_YIELD_KG_PER_HA AS start_yield,
       a.WHEAT_YIELD_KG_PER_HA AS end_yield,
       a.WHEAT_YIELD_KG_PER_HA - b.WHEAT_YIELD_KG_PER_HA AS yield_increase
FROM agri_data a
JOIN agri_data b
  ON a.DIST_CODE = b.DIST_CODE
 AND b.YEAR = a.YEAR - 5
WHERE a.YEAR = (SELECT MAX(YEAR) FROM agri_data)
  AND a.WHEAT_YIELD_KG_PER_HA > 0
  AND b.WHEAT_YIELD_KG_PER_HA > 0
ORDER BY yield_increase DESC
LIMIT 5
"""
df2 = run_query(q2)
display(df2)
px.bar(df2, x="district", y="yield_increase", color="state",
       title="Wheat yield increase (kg/ha) over 5 years").show()

,district,state,start_yield,end_yield,yield_increase
0,Vidisha,Madhya Pradesh,1754.10,3900.00,2145.90
1,Darjeeling,West Bengal,1172.59,3011.63,1839.04
2,Shajapur,Madhya Pradesh,2064.96,3748.81,1683.85
3,Damoh,Madhya Pradesh,2037.73,3500.00,1462.27
4,Ujjain,Madhya Pradesh,3040.96,4242.02,1201.06


3. States with the Highest Growth in Oilseed Production (5-Year Growth Rate)

In [28]:
q3 = """
SELECT STATE_NAME AS state,
       SUM(CASE WHEN YEAR = m.max_yr     THEN OILSEEDS_PRODUCTION_1000_TONS END) AS recent_prod,
       SUM(CASE WHEN YEAR = m.max_yr - 5 THEN OILSEEDS_PRODUCTION_1000_TONS END) AS old_prod,
       ROUND(
         (SUM(CASE WHEN YEAR = m.max_yr     THEN OILSEEDS_PRODUCTION_1000_TONS END)
        - SUM(CASE WHEN YEAR = m.max_yr - 5 THEN OILSEEDS_PRODUCTION_1000_TONS END))
         / NULLIF(SUM(CASE WHEN YEAR = m.max_yr - 5 THEN OILSEEDS_PRODUCTION_1000_TONS END), 0) * 100
       , 2) AS growth_rate_pct
FROM agri_data
CROSS JOIN (SELECT MAX(YEAR) AS max_yr FROM agri_data) m
WHERE YEAR IN (m.max_yr, m.max_yr - 5)
GROUP BY STATE_NAME
HAVING old_prod > 0
ORDER BY growth_rate_pct DESC
"""
df3 = run_query(q3)
display(df3)
px.bar(df3, x="state", y="growth_rate_pct",
       title="Oilseed production growth (%), 5-year").show()

,state,recent_prod,old_prod,growth_rate_pct
0,Andhra Pradesh,2876.52,1988.28,44.67
1,Haryana,1134.70,930.00,22.01
2,Orissa,535.34,689.87,-22.40
3,Chhattisgarh,0.00,223.94,-100.00
4,Telangana,0.00,806.62,-100.00
5,Karnataka,0.00,801.18,-100.00
6,Maharashtra,0.00,5159.94,-100.00
7,Gujarat,0.00,2854.89,-100.00
8,Rajasthan,0.00,6371.14,-100.00
9,Punjab,0.00,70.20,-100.00


4.District-wise Correlation Between Area and Production for Rice, Wheat, Maize

In [29]:
raw = run_query("""
SELECT DIST_NAME, STATE_NAME,
       RICE_AREA_1000_HA,  RICE_PRODUCTION_1000_TONS,
       WHEAT_AREA_1000_HA, WHEAT_PRODUCTION_1000_TONS,
       MAIZE_AREA_1000_HA, MAIZE_PRODUCTION_1000_TONS
FROM agri_data
""")
raw.columns = [c.upper() for c in raw.columns]

def district_corr(df, crop, min_years=10):
    area, prod = f"{crop}_AREA_1000_HA", f"{crop}_PRODUCTION_1000_TONS"
    d = df[(df[area] > 0) & (df[prod] > 0)]
    rows = []
    for (dist, state), g in d.groupby(["DIST_NAME", "STATE_NAME"]):
        if len(g) >= min_years:
            rows.append((dist, state, g[area].corr(g[prod])))
    return pd.DataFrame(rows, columns=["district", "state", f"{crop.lower()}_corr"])

df4 = (district_corr(raw, "RICE")
       .merge(district_corr(raw, "WHEAT"), on=["district", "state"], how="outer")
       .merge(district_corr(raw, "MAIZE"), on=["district", "state"], how="outer")
       .round(3))
display(df4.head(20))
display(df4[["rice_corr", "wheat_corr", "maize_corr"]].describe().round(3))
px.box(df4.melt(id_vars=["district", "state"], var_name="crop", value_name="correlation"),
       x="crop", y="correlation", title="District-level area vs production correlation").show()

,district,state,rice_corr,wheat_corr,maize_corr
0,24 Parganas,West Bengal,0.451,0.805,0.817
1,Adilabad,Telangana,0.815,0.382,0.684
2,Agra,Uttar Pradesh,0.978,0.863,0.505
3,Ahmedabad,Gujarat,0.937,0.890,0.890
4,Ahmednagar,Maharashtra,0.752,0.923,0.947
5,Ajmer,Rajasthan,0.889,0.628,0.029
6,Akola,Maharashtra,0.780,0.604,0.917
7,Alappuzha,Kerala,0.876,NaN,NaN
8,Aligarh,Uttar Pradesh,0.988,0.927,0.417
9,Allahabad,Uttar Pradesh,0.894,0.914,0.772


,rice_corr,wheat_corr,maize_corr
count,301.000,275.000,290.000
mean,0.655,0.798,0.729
std,0.330,0.222,0.278
min,-0.503,-0.247,-0.472
25%,0.514,0.762,0.603
50%,0.772,0.890,0.831
75%,0.905,0.936,0.940
max,0.996,1.000,0.999


5. Yearly Production Growth of Cotton in Top 5 Cotton Producing States

In [30]:
q5 = """
WITH top5 AS (
    SELECT STATE_NAME
    FROM agri_data
    GROUP BY STATE_NAME
    ORDER BY SUM(COTTON_PRODUCTION_1000_TONS) DESC
    LIMIT 5
),
yearly AS (
    SELECT a.YEAR AS year, a.STATE_NAME AS state,
           SUM(a.COTTON_PRODUCTION_1000_TONS) AS cotton_production
    FROM agri_data a
    JOIN top5 t ON a.STATE_NAME = t.STATE_NAME
    GROUP BY a.YEAR, a.STATE_NAME
)
SELECT year, state, cotton_production,
       ROUND(
         (cotton_production - LAG(cotton_production) OVER (PARTITION BY state ORDER BY year))
         / NULLIF(LAG(cotton_production) OVER (PARTITION BY state ORDER BY year), 0) * 100
       , 2) AS yoy_growth_pct
FROM yearly
ORDER BY state, year
"""
df5 = run_query(q5)
display(df5.head(10))
px.line(df5, x="year", y="yoy_growth_pct", color="state",
        title="Cotton production: year-over-year growth (%)").show()

,year,state,cotton_production,yoy_growth_pct
0,1966,Gujarat,252.0,NaN
1,1967,Gujarat,271.0,7.54
2,1968,Gujarat,264.0,-2.58
3,1969,Gujarat,295.3,11.86
4,1970,Gujarat,323.9,9.69
5,1971,Gujarat,451.4,39.36
6,1972,Gujarat,292.7,-35.16
7,1973,Gujarat,312.6,6.80
8,1974,Gujarat,254.2,-18.68
9,1975,Gujarat,316.6,24.55


6. Districts with the Highest Groundnut Production in the Latest Year

In [31]:
latest_year = int(run_query("SELECT MAX(YEAR) AS y FROM agri_data")["y"][0])
print("Latest year in data:", latest_year)

q6 = """
SELECT DIST_NAME AS district, STATE_NAME AS state,
       SUM(GROUNDNUT_PRODUCTION_1000_TONS) AS groundnut_production
FROM agri_data
WHERE YEAR = (SELECT MAX(YEAR) FROM agri_data)
  AND GROUNDNUT_PRODUCTION_1000_TONS > 0
GROUP BY DIST_NAME, STATE_NAME
ORDER BY groundnut_production DESC
LIMIT 10
"""
df6 = run_query(q6)
display(df6)
px.bar(df6, x="district", y="groundnut_production", color="state",
       title=f"Top 10 districts by groundnut production ({latest_year})").show()

Latest year in data: 2017


,district,state,groundnut_production
0,Jamnagar,Gujarat,977.65
1,Junagadh,Gujarat,945.49
2,Rajkot,Gujarat,873.87
3,Bikaner,Rajasthan,514.09
4,Ananthapur,Andhra Pradesh,454.94
5,Banaskantha,Gujarat,288.21
6,North Arcot / Vellore,Tamil Nadu,280.51
7,Mahabubnagar,Telangana,273.25
8,Chittoor,Andhra Pradesh,263.38
9,Sabarkantha,Gujarat,244.48


7. Annual Average Maize Yield Across All States

In [32]:
q7 = """
SELECT YEAR AS year, STATE_NAME AS state,
       ROUND(SUM(MAIZE_PRODUCTION_1000_TONS) / SUM(MAIZE_AREA_1000_HA) * 1000, 2) AS maize_yield_kg_per_ha
FROM agri_data
WHERE MAIZE_AREA_1000_HA > 0 AND MAIZE_PRODUCTION_1000_TONS > 0
GROUP BY YEAR, STATE_NAME
ORDER BY YEAR, STATE_NAME
"""
df7 = run_query(q7)
display(df7.head(10))

# All-India yearly figure, same weighting
q7b = """
SELECT YEAR AS year,
       ROUND(SUM(MAIZE_PRODUCTION_1000_TONS) / SUM(MAIZE_AREA_1000_HA) * 1000, 2) AS india_maize_yield_kg_per_ha
FROM agri_data
WHERE MAIZE_AREA_1000_HA > 0 AND MAIZE_PRODUCTION_1000_TONS > 0
GROUP BY YEAR
ORDER BY YEAR
"""
df7b = run_query(q7b)
px.line(df7b, x="year", y="india_maize_yield_kg_per_ha", title="All-India maize yield (kg/ha)").show()
px.line(df7, x="year", y="maize_yield_kg_per_ha", color="state", title="Maize yield by state (kg/ha)").show()

,year,state,maize_yield_kg_per_ha
0,1966,Andhra Pradesh,788.89
1,1966,Assam,547.27
2,1966,Bihar,1125.07
3,1966,Chhattisgarh,1303.80
4,1966,Gujarat,670.09
5,1966,Haryana,1042.74
6,1966,Himachal Pradesh,1519.53
7,1966,Jharkhand,1027.05
8,1966,Karnataka,801.05
9,1966,Madhya Pradesh,745.47


8. Total Area Cultivated for Oilseeds in Each State

In [33]:
q8 = """
WITH state_year AS (
    SELECT STATE_NAME, YEAR, SUM(OILSEEDS_AREA_1000_HA) AS area
    FROM agri_data
    WHERE OILSEEDS_AREA_1000_HA > 0
    GROUP BY STATE_NAME, YEAR
)
SELECT STATE_NAME AS state,
       ROUND(AVG(area), 2) AS avg_annual_area_1000ha,
       ROUND(SUM(CASE WHEN YEAR = (SELECT MAX(YEAR) FROM agri_data) THEN area END), 2) AS latest_year_area_1000ha
FROM state_year
GROUP BY STATE_NAME
ORDER BY avg_annual_area_1000ha DESC
"""
df8 = run_query(q8)
display(df8)
px.bar(df8, x="state", y="avg_annual_area_1000ha",
       title="Average annual oilseed area (1000 ha) by state").show()

,state,avg_annual_area_1000ha,latest_year_area_1000ha
0,Madhya Pradesh,3876.45,NaN
1,Rajasthan,2848.67,NaN
2,Maharashtra,2681.38,NaN
3,Gujarat,2614.51,NaN
4,Karnataka,1888.31,NaN
5,Andhra Pradesh,1677.37,914.70
6,Tamil Nadu,1147.46,NaN
7,Uttar Pradesh,878.51,NaN
8,Kerala,819.64,NaN
9,Orissa,763.12,603.37


9. Districts with the Highest Rice Yield

In [34]:
q9 = """
SELECT DIST_NAME AS district, STATE_NAME AS state,
       ROUND(AVG(RICE_YIELD_KG_PER_HA), 2) AS avg_rice_yield,
       COUNT(*) AS years_recorded
FROM agri_data
WHERE RICE_YIELD_KG_PER_HA > 0
GROUP BY DIST_NAME, STATE_NAME
HAVING COUNT(*) >= 10
ORDER BY avg_rice_yield DESC
LIMIT 10
"""
df9 = run_query(q9)
display(df9)
px.bar(df9, x="district", y="avg_rice_yield", color="state",
       title="Top 10 districts by average rice yield (kg/ha)").show()

,district,state,avg_rice_yield,years_recorded
0,Ludhiana,Punjab,3650.43,52
1,Sangrur,Punjab,3532.90,52
2,Thirunelveli,Tamil Nadu,3387.51,52
3,Bhatinda,Punjab,3363.16,52
4,Madurai,Tamil Nadu,3276.98,52
5,Kanyakumari,Tamil Nadu,3240.69,52
6,Ferozpur,Punjab,3234.06,52
7,Patiala,Punjab,3222.13,52
8,Jalandhar,Punjab,3187.03,52
9,Salem,Tamil Nadu,3102.58,52


10. Compare Production of Wheat and Rice for Top 5 States Over 10 Years

In [35]:
q10 = """
WITH bounds AS (
    SELECT MAX(YEAR) - 9 AS start_yr FROM agri_data
),
top5 AS (
    SELECT STATE_NAME
    FROM agri_data, bounds
    WHERE YEAR >= bounds.start_yr
    GROUP BY STATE_NAME
    ORDER BY SUM(RICE_PRODUCTION_1000_TONS) + SUM(WHEAT_PRODUCTION_1000_TONS) DESC
    LIMIT 5
)
SELECT a.YEAR AS year, a.STATE_NAME AS state,
       SUM(a.RICE_PRODUCTION_1000_TONS)  AS rice_production,
       SUM(a.WHEAT_PRODUCTION_1000_TONS) AS wheat_production
FROM agri_data a
JOIN top5 t ON a.STATE_NAME = t.STATE_NAME
CROSS JOIN bounds b
WHERE a.YEAR >= b.start_yr
GROUP BY a.YEAR, a.STATE_NAME
ORDER BY a.YEAR, a.STATE_NAME
"""
df10 = run_query(q10)
display(df10.head(10))
long10 = df10.melt(id_vars=["year", "state"], var_name="crop", value_name="production")
px.line(long10, x="year", y="production", color="state", line_dash="crop",
        title="Rice vs wheat production (1000 tons), top 5 states, last 10 years").show()

,year,state,rice_production,wheat_production
0,2008,Haryana,3299.00,11360.00
1,2008,Madhya Pradesh,1036.93,7272.20
2,2008,Punjab,11000.00,15733.00
3,2008,Uttar Pradesh,13080.93,29110.93
4,2008,West Bengal,15037.30,764.50
5,2009,Haryana,3628.00,10488.00
6,2009,Madhya Pradesh,1355.60,8865.30
7,2009,Punjab,11236.00,15169.00
8,2009,Uttar Pradesh,11840.15,27910.80
9,2009,West Bengal,14340.60,846.70
